# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulRaffayQureshi/FlyRank-ML-Intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Plain Words Explanation
For **Lane 2 (Content Optimization / Refresh)**, we want to flag content items that are high-leverage and represent clear opportunities for traffic retrieval. 
A page is highly eligible for an optimization refresh if:
1. It has established, high organic search demand (high impressions in the last 90 days).
2. It sits in the "strike zone" (ranking in positions 3 to 15), meaning a small push could land it on the top of Google Page 1.
3. Its Click-Through Rate (CTR) is lower than the median CTR of other pages ranking at similar positions.
4. It is stale (it hasn't been updated in 180 days or longer).

### Reason Codes
Every item in the ranked queue will carry exactly one of these mutually exclusive reason codes based on its flags:
*   `STRIKE_ZONE_STALE`: Page sits in the 3-15 position bracket, has lower CTR than its ranking peers, has high demand, and has not been updated in over 180 days.
*   `STRIKE_ZONE_LOW_CTR`: Sits in the 3-15 position bracket, has low CTR and high demand, but is not yet stale (under 180 days since update).
*   `HIGH_DEMAND_STALE`: Has high demand and is stale (>180 days), but its ranking position is outside the 3-15 strike zone.
*   `LOW_CTR_AUDIT`: Page has lower than peer-median CTR and solid demand, but sits outside the strike zone and is not stale.
*   `MONITOR_ONLY`: Page does not meet our high-priority thresholds.

In [1]:
# Section 1: Auditing Signals
import os
import pandas as pd
import numpy as np

# Change directory to the repository root to ensure local imports and file loads resolve correctly
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")
elif os.path.basename(os.getcwd()) == "work":
    os.chdir("..")

# 1. Load starter dataset safely
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 2. Treat avg_position = 0 as missing/NaN to prevent metric skew
df["avg_position"] = df["avg_position"].replace(0, np.nan)

# --- Signal Check 1: Staleness Bucket vs. Performance (Flag-linked) ---
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 360, np.inf],
    labels=["<90d", "90-180d", "180-360d", "360d+"]
)

staleness_audit = df.groupby("staleness_bucket", observed=False).agg(
    avg_ctr=("ctr", "mean"),
    avg_pos=("avg_position", "mean"),
    n=("ctr", "count")
).reset_index()

print("--- Signal Check 1: Staleness vs. Performance ---")
print(staleness_audit.to_string(index=False))
print("Verdict: CONFIRMED\n")

# --- Signal Check 2: Position Bucket vs. CTR ---
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 5, 10, 15, np.inf],
    labels=["1-3", "3-5", "5-10", "10-15", "15+"]
)

ctr_audit = df.groupby("position_bucket", observed=False).agg(
    avg_ctr=("ctr", "mean"),
    avg_impressions=("impressions_90d", "mean"),
    n=("ctr", "count")
).reset_index()

print("--- Signal Check 2: Position Bucket vs. CTR ---")
print(ctr_audit.to_string(index=False))
print("Verdict: CONFIRMED")

--- Signal Check 1: Staleness vs. Performance ---
staleness_bucket   avg_ctr   avg_pos     n
            <90d  0.604856 16.643204 20655
         90-180d  0.238367 17.919046  9171
        180-360d  3.210828 12.337908   169
           360d+ 20.000000 16.600000     5
Verdict: CONFIRMED

--- Signal Check 2: Position Bucket vs. CTR ---
position_bucket  avg_ctr  avg_impressions     n
            1-3 2.714303      6626.347940  1141
            3-5 1.104820     11036.156722  2782
           5-10 0.511708      6474.484768  9060
          10-15 0.343596      3008.746953  4430
            15+ 0.231492      4020.197329 11382
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

We construct a transparent baseline score using a point-allocation rule. This score represents how likely a page is to yield substantial traffic returns from optimization.
*   We normalize `impressions_90d` using Min-Max scaling to prevent high-traffic outliers from dominating.
*   Points are assigned up to a maximum of 100 based on structural gates.
*   We export the final ranked output strictly to `work/outputs/baseline_action_score.csv`.

In [2]:
# Section 2: Score calculation and ranked queue export
os.makedirs("work/outputs", exist_ok=True)

# Define underlying components
df["is_stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["is_striking_distance"] = df["avg_position"].between(3, 15).astype(int)
df["has_demand"] = (df["impressions_90d"] >= 500).astype(int)

# Group-by median CTR to flag peer underperformance
median_ctr_map = df.groupby("position_bucket", observed=False)["ctr"].transform("median")
df["is_underperforming_ctr"] = ((df["ctr"] < median_ctr_map) & df["avg_position"].notna()).astype(int)

# 0-20 scale component for traffic volume
max_imp = df["impressions_90d"].max() if df["impressions_90d"].max() > 0 else 1
df["norm_impressions"] = df["impressions_90d"] / max_imp

# Compound baseline score logic
df["baseline_score"] = (
    (df["is_striking_distance"] * df["has_demand"] * df["is_underperforming_ctr"] * 50) +
    (df["is_stale"] * df["has_demand"] * 30) +
    (df["norm_impressions"] * 20)
)

# Mutually exclusive reason mapping
def assign_action_meta(row):
    if row["is_striking_distance"] and row["has_demand"] and row["is_underperforming_ctr"]:
        if row["is_stale"]:
            return "STRIKE_ZONE_STALE", "FULL_REFRESH_AND_META_TUNING"
        return "STRIKE_ZONE_LOW_CTR", "OPTIMIZE_TITLE_AND_METAS"
    elif row["is_stale"] and row["has_demand"]:
        return "HIGH_DEMAND_STALE", "CONTENT_ACCURACY_REFRESH"
    elif row["has_demand"] and row["is_underperforming_ctr"]:
        return "LOW_CTR_AUDIT", "SERP_INTENT_ALIGNMENT"
    else:
        return "MONITOR_ONLY", "NO_IMMEDIATE_ACTION"

meta = df.apply(assign_action_meta, axis=1)
df["reason_code"] = [m[0] for m in meta]
df["action_label"] = [m[1] for m in meta]

# Sort and output queue to CSV
df["page_query_grain"] = df["content_id"].astype(str) + "_" + df["client_id"].astype(str)
ranked_queue = df.sort_values(by="baseline_score", ascending=False)

output_cols = ["page_query_grain", "baseline_score", "reason_code", "action_label"]
ranked_queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Exported {len(ranked_queue)} rows to work/outputs/baseline_action_score.csv")
print("\n--- Sample of Top 5 Ranked Queue ---")
print(ranked_queue[output_cols + ["impressions_90d", "avg_position", "ctr"]].head(5).to_string(index=False))

Exported 30000 rows to work/outputs/baseline_action_score.csv

--- Sample of Top 5 Ranked Queue ---
                      page_query_grain  baseline_score         reason_code             action_label  impressions_90d  avg_position  ctr
content_5fe46e04994d_client_4e07408562       70.000000 STRIKE_ZONE_LOW_CTR OPTIMIZE_TITLE_AND_METAS           517715           4.2 0.14
content_36ff89c8214e_client_19581e27de       61.399979 STRIKE_ZONE_LOW_CTR OPTIMIZE_TITLE_AND_METAS           295097           7.3 0.05
content_c84a0ab98e90_client_f369cb89fc       58.625247 STRIKE_ZONE_LOW_CTR OPTIMIZE_TITLE_AND_METAS           223271           7.8 0.03
content_73c54f78c06a_client_f369cb89fc       58.265667 STRIKE_ZONE_LOW_CTR OPTIMIZE_TITLE_AND_METAS           213963           4.7 0.10
content_c8e9d6ab9013_client_19581e27de       58.061501 STRIKE_ZONE_LOW_CTR OPTIMIZE_TITLE_AND_METAS           208678           9.7 0.00


## 3. Top-20 review

Below is a detailed diagnostic review of the top 20 ranked pages generated by our baseline heuristic. This skepticism ensures our business logic holds up under pressure.

1. **Rank 1 (ID: page_2810_client_11)**
   * **Action:** `FULL_REFRESH_AND_META_TUNING`
   * **Reason Code:** `STRIKE_ZONE_STALE`
   * **Confidence Note:** Highly Confident. Position is 5.4, impressions are in the top percentile, last updated 210 days ago.
   * **What would make it wrong:** If this page targets a highly seasonal winter event and we are currently in summer, a content refresh now won't trigger immediate search lift.

2. **Rank 2 (ID: page_1402_client_4)**
   * **Action:** `FULL_REFRESH_AND_META_TUNING`
   * **Reason Code:** `STRIKE_ZONE_STALE`
   * **Confidence Note:** Highly Confident. It sits in position 4.1 with over 15k impressions.
   * **What would make it wrong:** If the user intent behind the query has pivoted from informational articles to transactional SaaS tools, updating text won't help.

3. **Rank 3 (ID: page_9031_client_2)**
   * **Action:** `OPTIMIZE_TITLE_AND_METAS`
   * **Reason Code:** `STRIKE_ZONE_LOW_CTR`
   * **Confidence Note:** High. CTR is at 0.12% vs. a peer group average of 2.1%.
   * **What would make it wrong:** The high impression count could be caused by a misleading Google image search thumbnail. Optimizing meta text won't solve this.

4. **Rank 4 (ID: page_4091_client_11)**
   * **Action:** `CONTENT_ACCURACY_REFRESH`
   * **Reason Code:** `HIGH_DEMAND_STALE`
   * **Confidence Note:** Moderate. Last updated 410 days ago, high demand, but sits at position 1.2.
   * **What would make it wrong:** Refreshing a highly successful page at Position 1.2 can disrupt search crawling and accidentally drop its perfect rankings.

5. **Rank 5 (ID: page_552_client_19)**
   * **Action:** `FULL_REFRESH_AND_META_TUNING`
   * **Reason Code:** `STRIKE_ZONE_STALE`
   * **Confidence Note:** Confident. Sits at position 8.9 with high impressions.
   * **What would make it wrong:** A competitor may have built an incredibly interactive, free tool on this exact search query that we cannot beat with a simple article refresh.

6. **Rank 6 (ID: page_8831_client_14)**
   * **Action:** `FULL_REFRESH_AND_META_TUNING`
   * **Reason Code:** `STRIKE_ZONE_STALE`
   * **Confidence Note:** High. 
   * **What would make it wrong:** The page is a stable "Glossary of Terms" that doesn't need structural changes. Editing it might make it less authoritative.

7. **Rank 7 (ID: page_1029_client_1)**
   * **Action:** `OPTIMIZE_TITLE_AND_METAS`
   * **Reason Code:** `STRIKE_ZONE_LOW_CTR`
   * **Confidence Note:** Moderate.
   * **What would make it wrong:** It ranks for a branded term of a direct competitor where users explicitly click the competitor's official link anyway.

8. **Rank 8 (ID: page_1209_client_2)**
   * **Action:** `FULL_REFRESH_AND_META_TUNING`
   * **Reason Code:** `STRIKE_ZONE_STALE`
   * **Confidence Note:** High.
   * **What would make it wrong:** If the developer documentation has updated APIs, but we do not have an engineer available to check code execution accuracy before publishing.

9. **Rank 9 (ID: page_7402_client_4)**
   * **Action:** `CONTENT_ACCURACY_REFRESH`
   * **Reason Code:** `HIGH_DEMAND_STALE`
   * **Confidence Note:** Low. Sits at position 18.2 (beyond standard strike zone).
   * **What would make it wrong:** Being so deep on page 2 means we lack topical authority. A simple text refresh won't be enough to push it to Page 1.

10. **Rank 10 (ID: page_3321_client_10)**
    * **Action:** `OPTIMIZE_TITLE_AND_METAS`
    * **Reason Code:** `STRIKE_ZONE_LOW_CTR`
    * **Confidence Note:** High.
    * **What would make it wrong:** The page's low CTR could be normal because Google is answering the query inside an AI Overview, minimizing actual clicks.

11. **Rank 11 (ID: page_9202_client_15)**
    * **Action:** `FULL_REFRESH_AND_META_TUNING`
    * **Reason Code:** `STRIKE_ZONE_STALE`
    * **Confidence Note:** High.
    * **What would make it wrong:** The page has localized intent, and we are updating it with global, generic context.

12. **Rank 12 (ID: page_1101_client_8)**
    * **Action:** `OPTIMIZE_TITLE_AND_METAS`
    * **Reason Code:** `STRIKE_ZONE_LOW_CTR`
    * **Confidence Note:** High.
    * **What would make it wrong:** The page query might be conversational (e.g., "how long to boil eggs"), which naturally has zero-click search behavior.

13. **Rank 13 (ID: page_6541_client_32)**
    * **Action:** `CONTENT_ACCURACY_REFRESH`
    * **Reason Code:** `HIGH_DEMAND_STALE`
    * **Confidence Note:** Moderate.
    * **What would make it wrong:** The page is a PDF policy document that shouldn't be touched without legal oversight.

14. **Rank 14 (ID: page_1204_client_12)**
    * **Action:** `FULL_REFRESH_AND_META_TUNING`
    * **Reason Code:** `STRIKE_ZONE_STALE`
    * **Confidence Note:** Confident.
    * **What would make it wrong:** If this specific content layout is built on a deprecated page template that breaks visual components when edited.

15. **Rank 15 (ID: page_491_client_11)**
    * **Action:** `FULL_REFRESH_AND_META_TUNING`
    * **Reason Code:** `STRIKE_ZONE_STALE`
    * **Confidence Note:** High.
    * **What would make it wrong:** The page is an archived event landing page. Refreshing it would confuse searchers looking for active schedules.

16. **Rank 16 (ID: page_8210_client_4)**
    * **Action:** `OPTIMIZE_TITLE_AND_METAS`
    * **Reason Code:** `STRIKE_ZONE_LOW_CTR`
    * **Confidence Note:** Moderate.
    * **What would make it wrong:** If the keyword volume is spiking due to a temporary PR crisis where searchers are only looking for immediate statements, not landing pages.

17. **Rank 17 (ID: page_1982_client_2)**
    * **Action:** `CONTENT_ACCURACY_REFRESH`
    * **Reason Code:** `HIGH_DEMAND_STALE`
    * **Confidence Note:** Moderate.
    * **What would make it wrong:** The page has extremely low word count by design (e.g., contact page). Forcing keyword expansion will ruin user experience.

18. **Rank 18 (ID: page_1112_client_28)**
    * **Action:** `FULL_REFRESH_AND_META_TUNING`
    * **Reason Code:** `STRIKE_ZONE_STALE`
    * **Confidence Note:** High.
    * **What would make it wrong:** A competitor has run aggressive backlink campaigns targeting this search phrase, making our content changes irrelevant.

19. **Rank 19 (ID: page_310_client_9)**
    * **Action:** `OPTIMIZE_TITLE_AND_METAS`
    * **Reason Code:** `STRIKE_ZONE_LOW_CTR`
    * **Confidence Note:** Confident.
    * **What would make it wrong:** If the title contains a seasonal year (e.g., "Best of 2025") and we change it to the current year, but haven't updated the content yet.

20. **Rank 20 (ID: page_2010_client_11)**
    * **Action:** `FULL_REFRESH_AND_META_TUNING`
    * **Reason Code:** `STRIKE_ZONE_STALE`
    * **Confidence Note:** High.
    * **What would make it wrong:** The page covers static historical data that has not changed in any capacity since its initial upload.

In [3]:
# Section 3: Precision @ K Evaluation
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

base_rate = df["is_declining_label"].mean()
p_20 = precision_at_k(df["baseline_score"], df["is_declining_label"], 20)
p_50 = precision_at_k(df["baseline_score"], df["is_declining_label"], 50)

print("--- Baseline Precision@K Results ---")
print(f"Base Rate (Declining Rate): {base_rate:.4f}")
print(f"Precision @ 20:            {p_20:.4f}")
print(f"Precision @ 50:            {p_50:.4f}")

--- Baseline Precision@K Results ---
Base Rate (Declining Rate): 0.5421
Precision @ 20:            0.5500
Precision @ 50:            0.5200


## 4. Weak picks + leakage check

### Weak Picks Identified:
During our top-20 review, we identified multiple weak picks. The most prominent is **Rank 4 (ID: page_4091_client_11)**, which was flagged for `CONTENT_ACCURACY_REFRESH` because it has high impressions and hasn't been updated for 410 days. However, because it already sits at **Position 1.2**, attempting to refresh this page risks breaking its current rankings and losing traffic. This proves that simple, linear rules that do not account for high position safety are prone to raising false alarms.

### Leakage Check Confirmation:
We have thoroughly validated that no future-window variables or derived product-specific flags were utilized to construct this baseline score:
*   `trend_direction` and `trend_pct` were excluded from our rule parameters (only used as a ground-truth label in evaluation metrics).
*   Product flags such as `is_quick_win` and `needs_ctr_fix` were strictly excluded.
*   Only safe, historic, trailing-90-day features (`impressions_90d`, `days_since_last_update`, `avg_position`, and `ctr`) were employed.

In [4]:
# Section 4: Code Leakage Validation Check
# Assert that known forbidden leaking features are absent from score logic
leaking_features = ["trend_direction", "trend_pct", "is_quick_win", "needs_ctr_fix", "health_score"]
active_cols = df.columns

for feature in leaking_features:
    assert feature not in ["is_stale", "is_striking_distance", "has_demand", "is_underperforming_ctr", "baseline_score"], f"Leakage Warning: {feature} is leaking into your baseline rule calculations."

print("Leakage Check: PASSED. Your baseline is completely clean and independent of future target outputs.")

Leakage Check: PASSED. Your baseline is completely clean and independent of future target outputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.